In [79]:
categorical_cols = [
    'visit_type',
    'specialty_group',
    'reminder_channel',
    'sex_at_birth',
    'race_ethnicity_group',
    'preferred_language',
    'facility_type',
    'urban_rural_group',
    'payer_group',
    'service_line_focus',
    'weather_severity_group',
    'facility_name'
]
numerical_cols = [
    'lead_days',
    'reminder_days_before',
    'distance_to_clinic_miles',
    'patient_age',
    'sdoh_risk_score_x',
    'sdoh_risk_score_y',
    'prior_completed_visits_12m',
    'prior_no_show_count_12m',
    'prior_cancel_count_12m',
    'chronic_condition_count'
]# deprivation_quintile==service_line_focus

In [80]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score




merged_date = pd.read_csv('../data/clean/merged_data_cleaned.csv')
target = merged_date['no_show_flag']
features = merged_date.drop(columns=[
    'no_show_flag', 'appointment_id', 'patient_id', 'provider_id',
    'facility_id', 'pcp_provider_id', 'zip_area_id_x', 'zip_area_id_y', 
    'zip_area_id', 'month_num', 'weekday_num','payer_id','payer_primary_id'
])

x_train, x_test, y_train, y_test = train_test_split( features, target, test_size=0.2, random_state=42, stratify=target)

scaler = StandardScaler()
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

train_encoded_df = encoder.fit_transform(x_train[categorical_cols])
test_encoded_df = encoder.transform(x_test[categorical_cols])

x_train[numerical_cols] = scaler.fit_transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])

x_train = x_train.drop(columns=categorical_cols)
x_test = x_test.drop(columns=categorical_cols)

x_train = x_train.join(train_encoded_df)
x_test = x_test.join(test_encoded_df)

# type(train_encoded_df)
# type(x_train)



In [81]:

baseline = DummyClassifier(strategy='most_frequent') # accuracy = 86
# baseline = DummyClassifier(strategy='stratified', random_state=42) accuracy = 76

baseline.fit(x_train, y_train)
baseline_predictions = baseline.predict(x_test)


In [82]:
accuracy = accuracy_score(y_test, baseline_predictions)
print("Baseline Accuracy:", accuracy)

Baseline Accuracy: 0.8635522298334151


In [83]:
x_train.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['appointment_weekday', 'lead_days', 'reminder_sent_flag',
       'reminder_days_before', 'same_day_flag', 'telehealth_flag',
       'distance_to_clinic_miles', 'transportation_barrier_flag',
       'interpreter_needed_flag_x', 'patient_age',
       ...
       'facility_name_Riverbend East Urgent Care',
       'facility_name_Riverbend Endocrinology and Metabolism',
       'facility_name_Riverbend Midtown Internal Medicine',
       'facility_name_Riverbend North Family Medicine',
       'facility_name_Riverbend Orthopedics and Sports Medicine',
       'facility_name_Riverbend Pulmonary and Sleep Medicine',
       'facility_name_Riverbend South Family Medicine',
       'facility_name_Riverbend West Primary Care',
       'facility_name_Riverbend West Urgent Care',
       'facility_name_Riverbend Women's Health'],
      dtype='str', length=105)>

In [84]:
# decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

decision_tree_algo = DecisionTreeClassifier(random_state=42)
decision_tree_algo.fit(x_train,y_train)
decision_tree_predictions= decision_tree_algo.predict(x_test)

accuracy = accuracy_score(y_test, decision_tree_predictions)
print(accuracy)
con_matrix = confusion_matrix(y_test,decision_tree_predictions)
print(con_matrix)
report = classification_report(y_test,decision_tree_predictions)
print(report)

1.0
[[54586     0]
 [    0  8625]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     54586
           1       1.00      1.00      1.00      8625

    accuracy                           1.00     63211
   macro avg       1.00      1.00      1.00     63211
weighted avg       1.00      1.00      1.00     63211



In [85]:
for col in x_train.columns:
    if x_train[col].astype(str).eq('Riverbend West Primary Care').any():
        print(col)

In [86]:
print(features['facility_name'].unique())


<StringArray>
[              'Riverbend Behavioral Health',
               'Riverbend East Primary Care',
                'Riverbend West Urgent Care',
           'Riverbend South Family Medicine',
           'Riverbend North Family Medicine',
               'Riverbend West Primary Care',
        'Riverbend Downtown Family Medicine',
       'Riverbend Midtown Internal Medicine',
                  'Riverbend Women's Health',
                'Riverbend East Urgent Care',
    'Riverbend Pulmonary and Sleep Medicine',
               'Riverbend Cardiology Center',
 'Riverbend Orthopedics and Sports Medicine',
    'Riverbend Endocrinology and Metabolism']
Length: 14, dtype: str
